In [5]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import roc_auc_score, classification_report

# 1. Load data
print("Loading data...")
train_df = pd.read_csv("/content/train.csv")

# 2. Text cleaning function
def clean_text(text):
    text = text.lower() # Convert to lowercase
    text = re.sub(r"\'s", " is ", text) # Expand basic contractions
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text) # Remove special characters and punctuation
    text = re.sub(r"\s+", " ", text).strip() # Remove extra spaces
    return text

print("Cleaning text data...")
train_df['clean_text'] = train_df['comment_text'].apply(clean_text)
display(train_df[['comment_text', 'clean_text']].head(3))

Loading data...
Cleaning text data...


,comment_text,clean_text
0,Explanation\nWhy the edits made under my usern...,explanation why the edits made under my userna...
1,D'aww! He matches this background colour I'm s...,d aww he matches this background colour i m se...
2,"Hey man, I'm really not trying to edit war. It...",hey man i m really not trying to edit war it i...


In [6]:
# 3. Define features (X) and targets (y)
labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
X = train_df['clean_text']
y = train_df[labels]

# Split the dataset: 80% for training, 20% for validation to prevent data leakage
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")

Training samples: 127656
Validation samples: 31915


In [7]:
# 4. Text Vectorization (Convert words to numerical features)
# Keep the top 10,000 frequent words and remove standard English stop words
vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

# 5. Model Training
# We use OneVsRestClassifier because this is a Multi-Label classification task
print("Training Logistic Regression Baseline...")
model = OneVsRestClassifier(LogisticRegression(solver='liblinear'))
model.fit(X_train_tfidf, y_train)
print("Training complete!")

Training Logistic Regression Baseline...
Training complete!


In [8]:
# 6. Predictions and Evaluation
y_pred_prob = model.predict_proba(X_val_tfidf)
y_pred = model.predict(X_val_tfidf)

# Calculate Macro ROC-AUC - the primary metric for this dataset
roc_auc = roc_auc_score(y_val, y_pred_prob, average='macro')
print(f"Baseline Macro ROC-AUC Score: {roc_auc:.4f}\n")

# Detailed classification report (Precision, Recall, F1-score per class)
print("Classification Report:\n")
print(classification_report(y_val, y_pred, target_names=labels))

Baseline Macro ROC-AUC Score: 0.9765

Classification Report:

               precision    recall  f1-score   support

        toxic       0.91      0.61      0.73      3056
 severe_toxic       0.58      0.27      0.37       321
      obscene       0.92      0.63      0.74      1715
       threat       0.62      0.14      0.22        74
       insult       0.83      0.50      0.62      1614
identity_hate       0.74      0.15      0.25       294

    micro avg       0.88      0.55      0.68      7074
    macro avg       0.77      0.38      0.49      7074
 weighted avg       0.87      0.55      0.67      7074
  samples avg       0.06      0.05      0.05      7074



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
